
## الهدف
حساب نتائج تقييم نظام سَنَد من درجات المقيّمين، بطريقة موحّدة لكل معيار.

## المدخلات
- **ملف المقيّم الأول** و**ملف المقيّم الثاني**: نفس الأسئلة (68 سؤالًا) مع درجات كل معيار.

## الخطوات

**Correctness:** متوسط درجات المقيّمين، أو الدرجات النهائية إذا وُجد الملف النهائي.

4. **الاتفاق بين المقيّمين:** Cohen's kappa، وWeighted kappa للمعيار الترتيبي (Correctness).
5. **الحفظ:** ملخص النتائج وقائمة الاختلافات في ملفات CSV.

## حالة النتائج
كل رقم يعرض حالته:
- **نهائية:** حُسب من الدرجات النهائية بعد الحسم.


In [ ]:
# الخلية 1: التثبيت (مرة واحدة)
!pip -q install pandas scikit-learn

In [ ]:
# الخلية 2: رفع الملفات
from google.colab import files
uploaded = files.upload()
print("الملفات المرفوعة:", list(uploaded.keys()))

Saving grader_2_grading.csv to grader_2_grading.csv
Saving grader_1_grading.csv to grader_1_grading.csv
الملفات المرفوعة: ['grader_2_grading.csv', 'grader_1_grading.csv']


In [ ]:
# الخلية 3: الإعدادات
import os
G1 = "grader_1_grading.csv"
G2 = next((f for f in uploaded if f.startswith("grader_2")), "grader_2_grading.csv")
FINAL = next((f for f in uploaded if f.startswith("final")), None)   # اختياري
print("المقيّم 1:", G1, "| المقيّم 2:", G2, "| النهائي:", FINAL or "غير موجود (النتائج مؤقتة)")

المقيّم 1: grader_1_grading.csv | المقيّم 2: grader_2_grading.csv | النهائي: غير موجود (النتائج مؤقتة)


In [ ]:
import pandas as pd

def _load(p):
    for enc in ("utf-8-sig", "cp1256", "utf-8"):
        for sep in (";", ","):
            try:
                d = pd.read_csv(p, encoding=enc, sep=sep, engine="python")
                if "question_id" in d.columns and d.shape[1] >= 8:
                    return d
            except Exception:
                pass
    raise ValueError(f"تعذّرت قراءة {p}")

a = _load(G1)
b = _load(G2)
g1 = a["grader_correctness_0_2"].astype(float)
g2 = b["grader_correctness_0_2"].astype(float)
avg = (g1 + g2) / 2

def _row(name, s):
    return {"المصدر": name,
            "المتوسط (من 2)": round(s.mean(), 2),
            "Correctness %": f"{s.mean()/2:.1%}",
            "إجابات كاملة (2)": f"{int((s==2).sum())}/{len(s)}",
            "إجابات خاطئة (0)": f"{int((s==0).sum())}/{len(s)}"}

pd.DataFrame([_row("المقيّم 1", g1), _row("المقيّم 2", g2), _row("المتوسط بين المقيّمين", avg)])

,المصدر,المتوسط (من 2),Correctness %,إجابات كاملة (2),إجابات خاطئة (0)
0,المقيّم 1,1.81,90.4%,60/68,5/68
1,المقيّم 2,1.93,96.3%,65/68,2/68
2,المتوسط بين المقيّمين,1.87,93.4%,60/68,2/68


In [ ]:
summary_tbl = pd.DataFrame([_row("المقيّم 1", g1), _row("المقيّم 2", g2), _row("المتوسط بين المقيّمين", avg)])
summary_tbl.to_csv("correctness_summary.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("correctness_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>